# Flux Variability Analysis (FVA) on TCA Cycle Model

This notebook loads the `TCA_cycle` model, applies specific flux constraints with a 10% range, and performs FVA.

In [1]:
import cobra
from cobra import Model, Reaction, Metabolite
from cobra.flux_analysis import flux_variability_analysis
import pandas as pd
import numpy as np
from scipy.stats.qmc import LatinHypercube
import os

In [2]:
# Load the model
model_path = "../../models/TCA_cycle.json"
model = cobra.io.load_json_model(model_path)
print(f"Loaded model '{model.id}' with {len(model.reactions)} reactions and {len(model.metabolites)} metabolites.")

Loaded model 'metabolic_network' with 32 reactions and 22 metabolites.


## Apply Flux Constraints

We fix the following fluxes with a range of +/- 10%:
- R16 (PYR_B): 0.0700
- R17 (AKG_B): 0.2300
- R18 (OAA_B): 0.1200
- R19 (VALX): 0.0500
- R20 (LYSX): 0.0300

In [3]:
# R1 (PYR input) is the reference flux, fixed at 1000 (v1 = 1 in normalised units)
R1_REF = 1000.0
model.reactions.get_by_id("R1").bounds = (R1_REF, R1_REF)

# Scale the measured efflux targets by R1_REF so that the cobra model is
# consistent with the normalised flux_analysis.ipynb (where v1 = 1).
targets = {
    'R16': 0.0700 * R1_REF,   # PYR_B
    'R17': 0.2300 * R1_REF,   # AKG_B
    'R18': 0.1200 * R1_REF,   # OAA_B
    'R19': 0.0500 * R1_REF,   # VALX
    'R20': 0.0300 * R1_REF,   # LYSX
}

print(f"Reference flux R1 = {R1_REF}")
print("Applying constraints (Target ± 10%):")
for rxn_id, value in targets.items():
    delta = value * 0.10
    lb, ub = value - delta, value + delta
    model.reactions.get_by_id(rxn_id).bounds = (lb, ub)
    print(f"  {rxn_id}: Target={value:.1f}, Range=[{lb:.1f}, {ub:.1f}]")

Reference flux R1 = 1000.0
Applying constraints (Target ± 10%):
  R16: Target=70.0, Range=[63.0, 77.0]
  R17: Target=230.0, Range=[207.0, 253.0]
  R18: Target=120.0, Range=[108.0, 132.0]
  R19: Target=50.0, Range=[45.0, 55.0]
  R20: Target=30.0, Range=[27.0, 33.0]


## Perform Flux Variability Analysis

In [45]:
pd.set_option('display.max_rows', None)
# Run FVA
# fraction_of_optimum=0.0 allows exploring the full feasible space given constraints
fva_result = flux_variability_analysis(model, fraction_of_optimum=0.0, processes=1)
print("\nFVA Results:")
print(fva_result)


FVA Results:
             minimum    maximum
R1         1000.0000  1000.0000
R2            0.0000   330.1429
R3          153.7143   351.8000
R4          153.7143   351.8000
R5          153.7143   351.8000
R6            0.0000   230.8571
R7            0.0000  1000.0000
R8            0.0000  1000.0000
R9            0.0000   571.8000
R10         480.1818  1000.0000
R11         230.6667   450.6000
R12           0.0000   115.4286
R13         230.6667   500.0000
R14           0.0000   462.2000
R15           0.0000  1000.0000
R16          63.0000    77.0000
R17         207.0000   253.0000
R18         108.0000   132.0000
R19          45.0000    55.0000
R20          27.0000    33.0000
R21          10.2000   562.4286
EX_PYR_EX -1000.0000 -1000.0000
EX_GLU_EX  -330.1429     0.0000
EX_O2_EX   -500.0000  -230.6667
EX_CO2_EX     0.0000   462.2000
DM_PYR_B     63.0000    77.0000
DM_AKG_B    207.0000   253.0000
DM_OAA_B    108.0000   132.0000
DM_VALX      45.0000    55.0000
DM_LYSX      27.0000    33

In [47]:
# FVA feasible ranges for the three free fluxes
fva_ranges = {
    'R2': (fva_result.loc['R2', 'minimum'], fva_result.loc['R2', 'maximum']),
    'R8': (fva_result.loc['R8', 'minimum'], fva_result.loc['R8', 'maximum']),
    'R9': (fva_result.loc['R9', 'minimum'], fva_result.loc['R9', 'maximum']),
}
print("FVA ranges (absolute fluxes):")
for rxn, (lb, ub) in fva_ranges.items():
    print(f"  {rxn}: [{lb:.4f}, {ub:.4f}]")

# Latin Hypercube Sampling — 50 space-filling combinations
sampler = LatinHypercube(d=3, seed=42)
lhs_unit = sampler.random(n=50)   # shape (50, 3) in [0, 1]

bounds = list(fva_ranges.values())
scaled = np.column_stack([
    lhs_unit[:, i] * (ub - lb) + lb
    for i, (lb, ub) in enumerate(bounds)
])

flux_combinations = pd.DataFrame(scaled, columns=['R2', 'R8', 'R9'])
flux_combinations.index = range(1, 51)
flux_combinations.index.name = 'Sample'

# Divide by 1000 to match the flux scale used in flux_analysis.ipynb (v1 = 1)
flux_combinations_scaled = flux_combinations / 1000

out_path = "flux_samples_R2_R8_R9.csv"
flux_combinations_scaled.to_csv(out_path)
print(f"\nSaved {len(flux_combinations_scaled)} samples (÷ 1000) → {out_path}")
print(flux_combinations_scaled.round(4))

FVA ranges (absolute fluxes):
  R2: [0.0000, 330.1429]
  R8: [0.0000, 1000.0000]
  R9: [0.0000, 571.8000]

Saved 50 samples (÷ 1000) → flux_samples_R2_R8_R9.csv
           R2     R8     R9
Sample                     
1      0.1600 0.7312 0.1503
2      0.2199 0.2381 0.2633
3      0.1204 0.2643 0.2387
4      0.0300 0.1126 0.0008
5      0.2533 0.6635 0.4409
6      0.2956 0.3889 0.4681
7      0.0672 0.0074 0.3230
8      0.0835 0.8006 0.4129
9      0.3250 0.7961 0.4864
10     0.0063 0.6169 0.5640
11     0.1866 0.3206 0.0306
12     0.3079 0.7706 0.3867
13     0.1114 0.8705 0.4548
14     0.1012 0.1513 0.1735
15     0.2991 0.0338 0.4251
16     0.3116 0.2923 0.5342
17     0.2794 0.1772 0.0892
18     0.1518 0.5243 0.0953
19     0.1274 0.7044 0.3607
20     0.0227 0.4972 0.5133
21     0.0550 0.5506 0.3709
22     0.0742 0.1873 0.4969
23     0.1350 0.5739 0.5257
24     0.2282 0.4757 0.2469
25     0.0472 0.6353 0.2967
26     0.0378 0.9541 0.2097
27     0.2604 0.3443 0.4727
28     0.2680 0.8237 0.1125

In [5]:
from cobra.sampling import sample

# 1. APPLY CONSTRAINTS TO THE MODEL OBJECT
# Your loop correctly sets bounds. The sampler respects these.
for rxn_id, value in targets.items():
    delta = value * 0.10
    lb, ub = value - delta, value + delta
    model.reactions.get_by_id(rxn_id).bounds = (lb, ub)

# 2. RUN SAMPLING (The Correct Way)
# This generates full valid flux distributions for the WHOLE model
# 'achr' is standard for smaller models; 'optgp' is faster for huge models.
print("Starting polytope sampling...")
s = sample(model, 50, method="achr") 

# 3. EXTRACT THE COLUMNS YOU WANT
flux_combinations = s[['R2', 'R8', 'R9']]

# 4. NORMALIZE (Divide by 1000 to match v1=1 scale)
flux_combinations_scaled = flux_combinations / 1000.0

# 5. SAVE
out_path = "flux_samples_R2_R8_R9_validated.csv"
flux_combinations_scaled.to_csv(out_path)

print(f"\nSaved {len(flux_combinations_scaled)} valid samples.")
print(flux_combinations_scaled.head())

Starting polytope sampling...

Saved 50 valid samples.
         R2        R8        R9
0  0.113227  0.595017  0.324579
1  0.068030  0.751336  0.375948
2  0.034495  0.319359  0.117855
3  0.030391  0.443453  0.211884
4  0.148982  0.749244  0.144070


In [ ]:
from cobra.sampling import sample

# Load a fresh copy of the model — no efflux constraints applied
model_free = cobra.io.load_json_model("../../models/TCA_cycle.json")

# Only fix R1 as the reference flux (v1 = 1 in normalised units → 1000 absolute)
R1_REF = 1000.0
model_free.reactions.get_by_id("R1").bounds = (R1_REF, R1_REF)

print(f"Model: {len(model_free.reactions)} reactions, {len(model_free.metabolites)} metabolites")
print(f"R1 fixed at {R1_REF}. All other bounds left at model defaults.")

# Sample 1000 flux distributions across the full feasible polytope
print("Starting polytope sampling (1000 samples, ACHR)...")
samples_free = sample(model_free, 1000, method="achr")

# Normalise by R1_REF so fluxes are on the v1 = 1 scale
samples_free_scaled = samples_free / R1_REF

# Save
out_path_free = "flux_samples_unconstrained_1000.csv"
samples_free_scaled.index.name = "Sample"
samples_free_scaled.to_csv(out_path_free)

print(f"\nSaved {len(samples_free_scaled)} samples → {out_path_free}")
print(samples_free_scaled[['R2', 'R8', 'R9', 'R16', 'R17', 'R18', 'R19', 'R20']].describe().round(4))

## Unconstrained Network Sampling (R1 = 1000 only)

Sample 1000 flux solutions from the full feasible space with only R1 fixed as the reference flux. No constraints on effluxes (R16–R20).